In [ ]:
#change crest import to appropriate location
from crest import HierarchalGraphModel
import tensorflow as tf
import numpy as np
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

In [ ]:
num_train  = x_train.shape[0]
num_test   = x_test.shape[0]

img_height = x_train.shape[1]
img_width  = x_train.shape[2]
x_train = x_train.reshape((num_train, img_width * img_height))
x_test  = x_test.reshape((num_test, img_width * img_height))

y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

In [ ]:
#wrap dense layer to add a call that takes a dict
class KerasDenseLayer:
    """ Keras Dense Layer 
    
    input keys: x
    output keys : y
    
    """
    def __init__(self,name):
        self.dense = Dense(units=10,name='dense')
        self.name = 'dense'
        
    #re-write call to take a dictionary  
    def __call__(self,X):
        return {'y' : self.dense(X['x'])}

class DigitClassifier(HierarchalGraphModel):
    
    def __init__(self,name):
        super().__init__(None,name)
        self.num_classes = 10
        
        self.add_edge('input',KerasDenseLayer('dense'))
        self['dense'].input_features = 'x'
        self.add_edge('dense','output') 
        
        self.input_features = 'x'
        self.output_features = {'dense' : 'y'}

In [ ]:
#create Keras model
dc = DigitClassifier('dc')
x = Input(shape=(img_height*img_width,))
model = tf.keras.Model(inputs=x, outputs=dc({'x': x})['y'])

In [ ]:
#Compile and train using Keras APIs
model.compile(loss='categorical_crossentropy', 
              optimizer='adam',  
              metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=128,
          epochs=20,
          verbose=1,
          validation_split=0.1)

score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])